In [3]:
"""
Phidata ReAct-like agent + MySQL tools (schema, NL->SQL, execute)

Requires:
  - phidata>=2.7
  - sqlalchemy
  - pymysql
  - duckduckgo-search>=6.x
  - openai>=1.x
  - python-dotenv

Env you need:
  MYSQL_HOST, MYSQL_PORT, MYSQL_USER, MYSQL_PASSWORD, MYSQL_DB
  OPENAI_API_KEY
  (optional) MYSQL_MAX_ROWS
  (optional) OPENAI_NL2SQL_MODEL  # defaults to gpt-4o-mini
  (optional) OPENAI_AGENT_MODEL   # defaults to gpt-4o-mini
"""

from dotenv import load_dotenv

load_dotenv()

import os
import time
import re
from typing import Optional, List

from sqlalchemy import create_engine, text

# --- Phidata imports ---------------------------------------------------------
from phi.agent import Agent
from phi.model.openai import OpenAIChat
from phi.tools.duckduckgo import DuckDuckGo
from phi.tools import Toolkit

# --- OpenAI python client for NL->SQL ---------------------------------------
from openai import OpenAI


# ====================== Global agent system prompt ===========================

AGENT_SYSTEM_PROMPT = """
You are a helpful senior data assistant that talks to a MySQL database using tools.

Your main goals:
1. Understand the user's business question.
2. If the question does NOT contain enough specific information to build a precise SQL query,
   you MUST ask the user follow-up questions BEFORE calling any MySQL tools.

Examples of missing information you must clarify:
- Which table(s) to use, if it is ambiguous.
- Which experiment_id, user_id, or other identifiers to filter on.
- Required date range or time window (e.g. "last 7 days", "2024 only").
- Which metric(s) or columns to aggregate, and how (SUM / AVG / MAX / MIN / COUNT).
- Any grouping needed (e.g. "by experiment", "by day", "by dataset").
- Any filters for environment, status, tags, etc.

Guidelines:
- Do NOT guess table or column names.
- Do NOT assume default filters like experiment_id = 1 unless the user said so.
- When clarification is needed, ask the user directly in natural language.
- Ask for all obviously needed details in as few follow-up questions as possible.
- Only after you have the necessary details, use the MySQL tools to:
  (1) inspect schema if needed,
  (2) generate SQL from natural language,
  (3) execute the SQL and return the results.

If the user query is already specific enough to write an unambiguous SELECT query,
you can directly use the MySQL tools.
""".strip()


# ====================== Helpers =============================================

def _mysql_uri_from_env() -> str:
    host = os.getenv("MYSQL_HOST", "localhost")
    port = int(os.getenv("MYSQL_PORT", "3306"))
    user = os.getenv("MYSQL_USER")
    pwd = os.getenv("MYSQL_PASSWORD")
    db = os.getenv("MYSQL_DB")
    if not all([user, pwd, db]):
        raise RuntimeError("Set MYSQL_USER, MYSQL_PASSWORD, and MYSQL_DB in env")
    return f"mysql+pymysql://{user}:{pwd}@{host}:{port}/{db}"


_SELECT_ONLY = re.compile(r"(?is)^\s*select\b")


def _is_select_only(sql: str) -> bool:
    s = sql.strip()
    if ";" in s:
        return False
    return bool(_SELECT_ONLY.match(s))


# ====================== Custom MySQL Toolkit ================================

class MySQLToolkit(Toolkit):
    """
    Custom Toolkit exposing three tools to the Phidata Agent:

    1. mysql_schema(table_list: str = "") -> str
    2. mysql_nl2sql(question: str) -> str
    3. mysql_query_exec(sql: str) -> str
    """

    def __init__(self):
        super().__init__(name="mysql_tools")
        self._engine = None
        self._openai_client = OpenAI()

        # Register functions as tools so the Agent can call them
        self.register(self.mysql_schema)
        self.register(self.mysql_nl2sql)
        self.register(self.mysql_query_exec)

    # --- internal helpers ----------------------------------------------------

    @property
    def engine(self):
        if self._engine is None:
            self._engine = create_engine(
                _mysql_uri_from_env(),
                pool_pre_ping=True,
            )
        return self._engine

    # ====================== Tool 1: Schema discovery =========================

    def mysql_schema(self, table_list: str = "") -> str:
        """
        Discover MySQL schema.

        Input:
          - empty string: list all tables
          - OR a comma-separated list of tables.

        Returns a human-readable description of the tables and columns.
        """
        with self.engine.connect() as conn:
            if not table_list.strip():
                tables = [
                    r[0] for r in conn.execute(text("SHOW TABLES")).fetchall()
                ]
            else:
                tables = [t.strip() for t in table_list.split(",") if t.strip()]

            if not tables:
                return "No tables found."

            lines: List[str] = []
            for t in tables:
                try:
                    rows = conn.execute(text(f"DESCRIBE `{t}`")).fetchall()
                except Exception as e:
                    lines.append(f"# {t}\nERROR: {e}")
                    continue
                lines.append(f"# {t}")
                for field, col_type, nullable, key, default, extra in rows:
                    lines.append(
                        f"- {field}: {col_type}, NULL={nullable}, "
                        f"KEY={key}, DEFAULT={default}, EXTRA={extra}"
                    )

        return "\n".join(lines)

    # ====================== Tool 2: Natural language -> SQL ==================

    def mysql_nl2sql(self, question: str) -> str:
        """
        Generate ONE safe MySQL SELECT query from a natural-language question.

        Uses the database schema loaded from local file 'mlflow_db.txt'.
        Input: the user question (with all necessary details).
        Output: SQL only, one SELECT statement, no semicolons.
        """
        if not question or not question.strip():
            return "Please provide a question."

        # ---- Load schema from file -----------------------------------------
        try:
            with open("mlflow_db.txt", "r", encoding="utf-8") as f:
                schema_text = f.read()
        except Exception as e:
            return f"Could not read schema file 'mlflow_db.txt': {e}"

        system_prompt = (
            "You are a senior data analyst. Given a MySQL database schema and a "
            "natural language question, produce ONE valid MySQL SELECT query that "
            "answers the question using only the tables and columns described in the "
            "schema. No comments, no explanations, no DDL/DML, SELECT-only.\n"
            "- Always wrap table names and column names in backticks (`) if they "
            "contain spaces, special characters, or match MySQL reserved keywords.\n"
        )

        user_content = (
            "Database schema (from file mlflow_db.txt):\n"
            f"{schema_text}\n\n"
            "Question:\n"
            f"{question.strip()}\n\n"
            "Constraints:\n"
            "- Only one SELECT statement\n"
            "- No semicolons\n"
            "- MySQL dialect\n\n"
            "-- Always wrap table names and column names in backticks (`) if they "
            "contain spaces, special characters, or match MySQL reserved keywords.\n"
            "SQL:"
        )

        model_name = os.getenv("OPENAI_NL2SQL_MODEL", "gpt-4o-mini")

        resp = self._openai_client.chat.completions.create(
            model=model_name,
            temperature=0,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_content},
            ],
        )

        sql = (resp.choices[0].message.content or "").strip()

        # ---- Safety: enforce a single SELECT without semicolons ------------
        sql = re.sub(r";.*$", "", sql, flags=re.S).strip()
        if not _is_select_only(sql):
            return (
                "Failed to produce a safe single SELECT statement. "
                "Try rephrasing the question."
            )

        return sql

    # ====================== Tool 3: Execute SQL (read-only) ==================

    def mysql_query_exec(self, sql: str) -> str:
        """
        Execute a single SELECT query against MySQL and return rows (capped).

        Input must be ONE SELECT statement, no semicolons.
        Use this ONLY AFTER the SQL has been clearly defined/validated by the conversation.
        Do NOT guess or generate SQL inside this tool.
        """
        if not _is_select_only(sql):
            return "Only a single SELECT statement without ';' is allowed."

        row_cap = int(os.getenv("MYSQL_MAX_ROWS", "200"))
        with self.engine.connect() as conn:
            result = conn.execute(text(sql))
            rows = result.fetchmany(row_cap + 1)
            headers = list(result.keys())

        truncated = len(rows) > row_cap
        rows = rows[:row_cap]

        out: List[str] = []
        header_line = " | ".join(map(str, headers))
        out.append(header_line)
        out.append("-" * max(3, len(header_line)))
        for r in rows:
            out.append(" | ".join("" if v is None else str(v) for v in r))
        if truncated:
            out.append(f"...(truncated at {row_cap} rows)")
        return "\n".join(out)


# ====================== Agent builder =======================================

def build_mysql_agent(model_name: Optional[str] = None) -> Agent:
    """
    Create a Phidata Agent with DuckDuckGo + MySQL tools.

    The Agent:
      - Follows AGENT_SYSTEM_PROMPT for clarification rules.
      - Can search the web via DuckDuckGo.
      - Can inspect schema, generate SQL, and execute it in a read-only way.
    """
    llm_id = model_name or os.getenv("OPENAI_AGENT_MODEL", "gpt-4o-mini")

    model = OpenAIChat(
        id=llm_id,
        temperature=0,
    )

    mysql_tools = MySQLToolkit()
    web_search = DuckDuckGo()

    agent = Agent(
        model=model,
        tools=[web_search, mysql_tools],
        description=(
            "You are a helpful senior data assistant that talks to a MySQL database "
            "using tools."
        ),
        instructions=[AGENT_SYSTEM_PROMPT],
        markdown=True,
        show_tool_calls=True,
        add_datetime_to_instructions=True,
        # debug_mode=True,
    )
    return agent


# ====================== Interactive CLI demo ================================

if __name__ == "__main__":
    agent = build_mysql_agent()
    print("[demo] Phidata-based MySQL assistant is running.")
    print("Type your question (or 'exit' / 'quit' to stop).\n")

    while True:
        try:
            user_input = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n[demo] Exiting.")
            break

        if not user_input:
            continue
        if user_input.lower() in {"exit", "quit"}:
            print("[demo] Exiting.")
            break

        start = time.time()

        # --- KEY CHANGE: use run() instead of print_response() ---
        try:
            run_response = agent.run(user_input, stream=False)
        except Exception as e:
            print(f"\n[ERROR] Agent.run failed: {e}\n")
            continue

        # RunResponse.content holds the text answer
        content = getattr(run_response, "content", run_response)
        print(f"\nAssistant:\n{content}\n")

        elapsed = time.time() - start
        print(f"[demo] Turn completed in {elapsed:.2f}s\n")

[demo] Phidata-based MySQL assistant is running.
Type your question (or 'exit' / 'quit' to stop).



You:  provide description of each table in the database mlflow_db



Assistant:

Running:
 - mysql_schema()

Here is the description of each table in the `mlflow_db` database:

### 1. alembic_version
- **version_num**: varchar(32), primary key, stores the version number of the database schema.

### 2. datasets
- **dataset_uuid**: varchar(36), foreign key, unique identifier for the dataset.
- **experiment_id**: int(11), primary key, identifier for the associated experiment.
- **name**: varchar(500), primary key, name of the dataset.
- **digest**: varchar(36), primary key, a unique hash of the dataset.
- **dataset_source_type**: varchar(36), type of the dataset source.
- **dataset_source**: text, source location of the dataset.
- **dataset_schema**: mediumtext, schema of the dataset.
- **dataset_profile**: mediumtext, profile information of the dataset.

### 3. experiment_tags
- **key**: varchar(250), primary key, tag key for the experiment.
- **value**: varchar(5000), value associated with the tag.
- **experiment_id**: int(11), primary key, identifier f

You:  provide me description & counts of all columns of the table datasets and its signifcance



Assistant:

Running:
 - mysql_schema(table_list=datasets)

The `datasets` table contains the following columns:

| Column Name            | Data Type      | Null Allowed | Key Type | Default | Description |
|-----------------------|----------------|--------------|----------|---------|-------------|
| dataset_uuid          | varchar(36)    | NO           | MUL      | None    | A unique identifier for the dataset. |
| experiment_id         | int(11)        | NO           | PRI      | None    | The ID of the experiment associated with the dataset. |
| name                  | varchar(500)   | NO           | PRI      | None    | The name of the dataset. |
| digest                | varchar(36)    | NO           | PRI      | None    | A hash or digest of the dataset for integrity verification. |
| dataset_source_type   | varchar(36)    | NO           |          | None    | The type of source from which the dataset is derived (e.g., file, database). |
| dataset_source        | text           

You:  exit


[demo] Exiting.
